# UMAP 3D — supervised vs unsupervised (English-only subclass labels)

Loads pre-built files that already contain the `class` subclass field, so no in-notebook classifier training is needed:

- `crag_domain_open_ai_embeddings_subclass.json` + `crag_tfidf_embeddings_subclass.json` — CRAG, all English (`class` derived from CRAG `domain` + regex split of `open`).
- `chatbot_arena_*_sample3000_subclass_en.json` — chatbot_arena 3000-sample, English-only (2916 rows), `class` predicted offline by `predict_chatbot_arena_subclass.py`.

Up to 13 categories: `finance`/`movie`/`music`/`sports` + `celebrities`/`geography_places`/`art_history`/`business_companies`/`food`/`math_code`/`gaming`/`relationship_psychology` + `open` (fallback).

Visualize each embedding with UMAP in 3D:
- **unsupervised** — UMAP ignores labels (geometry only).
- **supervised by class** — `target_weight=0.001` nudges layout toward the subclass labels.
- **supervised by origin** — `target_weight=0.001` nudges layout toward crag=0 / chatbot_arena=1.

Total: 6 UMAP plots (OpenAI ×3 + TF-IDF ×3). Same outlier-drop + centering pipeline as `umap_embeddings_3d.ipynb`.

Install deps if missing:
```
python -m pip install umap-learn plotly scikit-learn pandas
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import umap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

## 0. Load all 4 files

In [ ]:
DATA_DIR = Path('/Users/s.petryshyn/Desktop/UNI/COURSE_WORK/data/processed')

FILES = {
    'crag_openai': (DATA_DIR / 'crag_domain_adapted_gold_draft_open_ai_embeddings.json',         'open_ai_embeddings'),
    'chat_openai': (DATA_DIR / 'chatbot_arena_open_ai_embeddings_sample3000_subclass_en.json',   'open_ai_embeddings'),
}

frames = {}
for key, (path, col) in FILES.items():
    df = pd.read_json(path).dropna(subset=['query']).reset_index(drop=True)
    df['query'] = df['query'].astype(str)
    frames[key] = df
    print(f'{key}: n={len(df)} cols={list(df.columns)} emb_dim={len(df[col].iloc[0])}')

## 1. Use the pre-built `class` labels

Both CRAG and chatbot_arena files already have a `class` field (CRAG = regex/native, chatbot_arena = LogReg prediction stored offline). Just copy it into `category` for the plot helper.

In [ ]:
for key in frames:
    frames[key]['category'] = frames[key]['class'].astype(str).str.lower()

print('crag_openai class dist:')
print(frames['crag_openai']['category'].value_counts())
print('\nchat_openai class dist (predicted):')
print(frames['chat_openai']['category'].value_counts())

## 2. UMAP 3D — unsupervised vs supervised

Helper builds the combined matrix (CRAG + chat), runs UMAP (optionally supervised with `target_weight`), drops top 1% outliers from the median, re-centers, returns a DataFrame ready to plot.

In [ ]:
# All helpers + plotting constants in one cell. Run this BEFORE any plot cell.
DROP_PCT = 1.0

# 13 subclasses. Order fixed so colors stay consistent across plots.
ALL_CATS = [
    'finance', 'movie', 'music', 'sports',
    'celebrities', 'geography_places', 'art_history', 'business_companies',
    'food', 'math_code', 'gaming', 'relationship_psychology',
    'open',
]
palette = px.colors.qualitative.Bold + px.colors.qualitative.Vivid
COLOR_MAP = {c: palette[i % len(palette)] for i, c in enumerate(ALL_CATS)}
SYMBOL = {'chatbot_arena': 'circle', 'crag': 'cross'}  # cross == '+'


def stack(df, col):
    return np.vstack(df[col].map(np.asarray).values).astype(np.float32)


def run_umap(crag_df, chat_df, emb_col, target_weight=None, y_col='category', label=''):
    crag_df = crag_df.copy(); crag_df['source'] = 'crag';          crag_df['origin'] = 0
    chat_df = chat_df.copy(); chat_df['source'] = 'chatbot_arena'; chat_df['origin'] = 1
    combined = pd.concat([crag_df, chat_df], ignore_index=True)
    X = stack(combined, emb_col)

    kwargs = dict(n_components=3, n_neighbors=20, min_dist=0.1, metric='cosine', random_state=42)
    if target_weight is not None:
        kwargs['target_weight'] = target_weight
        kwargs['target_metric'] = 'categorical'
    reducer = umap.UMAP(**kwargs)

    if target_weight is not None:
        y = LabelEncoder().fit_transform(combined[y_col].astype(str).values)
        emb3d = reducer.fit_transform(X, y=y)
    else:
        emb3d = reducer.fit_transform(X)

    center = np.median(emb3d, axis=0)
    dist = np.linalg.norm(emb3d - center, axis=1)
    cutoff = np.percentile(dist, 100 - DROP_PCT)
    keep = dist <= cutoff
    print(f'[{label}] dropped {(~keep).sum()} outliers, kept {keep.sum()}')

    out = combined.loc[keep, ['query', 'category', 'source']].reset_index(drop=True).copy()
    emb3d = emb3d[keep]
    center = np.median(emb3d, axis=0)
    out['umap_x'] = emb3d[:, 0] - center[0]
    out['umap_y'] = emb3d[:, 1] - center[1]
    out['umap_z'] = emb3d[:, 2] - center[2]
    return out


def plot_umap_3d(df, title, zoom=1.0, range_pct=100.0):
    """zoom>1 = closer camera. range_pct<100 = crop axes to percentile of |coords|."""
    fig = go.Figure()
    cats_in_df = list(df['category'].unique())
    cats = [c for c in ALL_CATS if c in cats_in_df] + [c for c in cats_in_df if c not in ALL_CATS]
    for source, sym in SYMBOL.items():
        for cat in cats:
            sub = df[(df['source'] == source) & (df['category'] == cat)]
            if len(sub) == 0:
                continue
            fig.add_trace(go.Scatter3d(
                x=sub['umap_x'], y=sub['umap_y'], z=sub['umap_z'],
                mode='markers',
                name=f'{source} | {cat}',
                marker=dict(
                    size=3 if source == 'chatbot_arena' else 5,
                    symbol=sym,
                    color=COLOR_MAP.get(cat, '#888888'),
                    opacity=0.75,
                    line=dict(width=0),
                ),
                hovertext=sub['query'].str.slice(0, 120),
                hoverinfo='text+name',
            ))

    coords_abs = np.abs(df[['umap_x', 'umap_y', 'umap_z']].values)
    if range_pct >= 100:
        axis_max = float(coords_abs.max()) * 1.05
    else:
        axis_max = float(np.percentile(coords_abs, range_pct)) * 1.02
    r = [-axis_max, axis_max]

    eye_v = 1.25 / max(zoom, 1e-6)  # default plotly eye is ~1.25; smaller = closer
    camera = dict(eye=dict(x=eye_v, y=eye_v, z=eye_v))

    fig.update_layout(
        title=title,
        width=1200, height=800,
        scene=dict(
            xaxis=dict(title='UMAP-1', range=r, zeroline=True),
            yaxis=dict(title='UMAP-2', range=r, zeroline=True),
            zaxis=dict(title='UMAP-3', range=r, zeroline=True),
            aspectmode='cube',
            camera=camera,
        ),
        legend=dict(itemsizing='constant', groupclick='toggleitem'),
    )
    return fig

print('helpers ready: stack, run_umap, plot_umap_3d')

### 2a. OpenAI — unsupervised

In [ ]:
df_openai_unsup = run_umap(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', target_weight=None, label='openai unsup')
fig_openai_unsup = plot_umap_3d(df_openai_unsup, 'UMAP 3D — OpenAI, unsupervised (dot=chatbot_arena, +=crag)')
fig_openai_unsup.show()

### 2b. OpenAI — supervised by class, target_weight=0.001

In [ ]:
df_openai_sup = run_umap(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', target_weight=0.001, y_col='category', label='openai sup class')
fig_openai_sup = plot_umap_3d(df_openai_sup, 'UMAP 3D — OpenAI, supervised by class (target_weight=0.001)')
fig_openai_sup.show()

### 2c. OpenAI — supervised by origin (crag=0, chat=1), target_weight=0.001

In [ ]:
df_openai_sup_origin = run_umap(frames['crag_openai'], frames['chat_openai'], 'open_ai_embeddings', target_weight=0.001, y_col='origin', label='openai sup origin')
fig_openai_sup_origin = plot_umap_3d(df_openai_sup_origin, 'UMAP 3D — OpenAI, supervised by origin (target_weight=0.001)')
fig_openai_sup_origin.show()

## 3. Save HTML (optional)

In [ ]:
for name, fig in [
    ('umap_3d_openai_unsup_subclass_en.html',             fig_openai_unsup),
    ('umap_3d_openai_sup_class_w0.001_subclass_en.html',  fig_openai_sup),
    ('umap_3d_openai_sup_origin_w0.001_subclass_en.html', fig_openai_sup_origin),
]:
    out = DATA_DIR / name
    fig.write_html(str(out))
    print('saved ->', out)